### Gold Layer - Create gold_olist_monthly_payment_insights using silver silver_olist_orders and silver_olist_orderPayment

In [0]:
# CREATE table gold_monthly_payment_insights
spark.sql("""
    CREATE OR REPLACE TABLE upskill.pyspark_learning.gold_olist_monthly_payment_insights AS
    SELECT
        order_purchase_year,
        order_purchase_month,
        payment_type,
        ROUND(SUM(op.payment_value),2) AS total_revenue,
        COUNT(DISTINCT o.order_id) AS total_orders
    FROM
        upskill.pyspark_learning.silver_olist_orders o
    JOIN 
        upskill.pyspark_learning.silver_olist_orderPayment op
        ON o.order_id = op.order_id
    WHERE
        o.order_status = 'delivered'
    GROUP BY 
        o.order_purchase_year,
        o.order_purchase_month,
        op.payment_type
    ORDER BY 
        o.order_purchase_year,
        o.order_purchase_month,
        total_revenue DESC 
""")


In [0]:
%sql
-- Write a SQL query using silver_orders and silver_order_payments (or querying gold_monthly_payment_insights) to compute:
-- 1. Total Monthly Delivered Revenue (sum across all payment types for each year & month).
-- 2. Previous Month's Revenue: Use the LAG() window function to retrieve the previous month's revenue for comparison.
-- 3. Month-over-Month (MoM) Growth Percentage 
--    MoM Growth % = (Current Month Revenue - Previous Month Revenue) / Previous Month Revenue * 100

WITH monthly_revenue AS (
    -- Step 1: Aggregate total revenue per year and month
    SELECT
        order_purchase_year,
        order_purchase_month,
        ROUND(SUM(total_revenue), 2) AS current_month_revenue
    FROM
        upskill.pyspark_learning.gold_olist_monthly_payment_insights
    GROUP BY    
        order_purchase_year,
        order_purchase_month
),

lagged_revenue AS (
    -- Step 2: Calculate previous month's revenue across years
    SELECT
        order_purchase_year,
        order_purchase_month,
        current_month_revenue,
        LAG(current_month_revenue) OVER (
            ORDER BY order_purchase_year, order_purchase_month
        ) AS prev_month_revenue
    FROM
        monthly_revenue
)

-- Step 3: Compute MoM growth percentage using clean aliases
SELECT
    order_purchase_year,
    order_purchase_month,
    current_month_revenue,
    prev_month_revenue,
    CASE 
        WHEN prev_month_revenue IS NULL OR prev_month_revenue = 0 THEN NULL
        ELSE ROUND(((current_month_revenue - prev_month_revenue) / prev_month_revenue) * 100, 2)
    END AS mom_growth_pct
FROM
    lagged_revenue
ORDER BY
    order_purchase_year,
    order_purchase_month         


### Create TABLE gold_state_customer_metrics using below tables:
    silver_olist_orders
    silver_olist_orderPayment
    silver_olist_customer

In [0]:
%sql
--Build a summary table called upskill.pyspark_learning.gold_state_customer_metrics that calculates:
--customer_state: The state code.total_delivered_orders: Total count of unique order_ids where order_status = 'delivered'.--total_unique_customers: Count of distinct customer_unique_ids who placed a delivered order.total_revenue: Total sum of payment_value --rounded to 2 decimal places.avg_order_value: Average revenue per order  = total_revenue / total_delivered\_orders (rounded to 2 decimal places)

CREATE OR REPLACE TABLE upskill.pyspark_learning.gold_gold_state_customer_metrics AS
    SELECT
        c.customer_state,
        count(DISTINCT o.order_id) AS total_delivered_orders,
        count(DISTINCT c.customer_unique_id) AS total_unique_customers,
        ROUND(sum(op.payment_value),2) AS total_revenue,
        ROUND((sum(op.payment_value) * 1.0 / count(DISTINCT o.order_id)),2) AS avg_order_value
    FROM
        upskill.pyspark_learning.silver_olist_orders o
    JOIN 
        upskill.pyspark_learning.silver_olist_orderPayment op
    ON
        o.order_id = op.order_id
    JOIN 
        upskill.pyspark_learning.silver_olist_customer c
    ON
        o.customer_id = c.customer_id
    WHERE
        o.order_status = 'delivered'
    GROUP BY 
        c.customer_state
    ORDER BY 
        total_revenue DESC


### Create gold table for  Top Product Categories Analysis

In [0]:
%sql
-- Top Product Categories Analysis (Spark SQL)

CREATE OR REPLACE TABLE upskill.pyspark_learning.gold_top_product_categories AS
SELECT 
    p.product_category,
    COUNT(oi.order_item_id) AS total_units_sold,
    ROUND(SUM(oi.price), 2) AS total_item_revenue,
    ROUND(AVG(oi.price), 2) AS avg_item_price
FROM
    upskill.pyspark_learning.silver_olist_products p
JOIN
    upskill.pyspark_learning.silver_olist_order_items oi
    ON p.product_id = oi.product_id
JOIN 
    upskill.pyspark_learning.silver_olist_orders o
    ON oi.order_id = o.order_id
WHERE
    o.order_status = 'delivered'
GROUP BY
    p.product_category
ORDER BY
    total_item_revenue DESC;